# PyTorch en une page

Une carte pour situer les outils avant d’ouvrir leurs détails.

Nous exécutons ce notebook avec un noyau Python neuf, sur CPU.

Version vérifiée : Python 3.11.9, PyTorch 2.10.0 et TorchAudio 2.10.0.

Les cellules n’installent aucun paquet et ne téléchargent aucune donnée.

Les nombres du cours viennent de cette version ; une autre version peut modifier certains arrondis ou messages.

Dans Colab, choisissons un environnement sans accélérateur, puis Exécution → Tout exécuter.

Les trous des exercices restent dans du Markdown : ils ne cassent donc pas l’exécution complète.

[Relire le chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#pytorch-en-une-page).

Le programme suivant est copié à l’identique depuis le cours.

In [1]:
import os
import sys
import tempfile

# À lancer dans un noyau neuf : ce tutoriel reste sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python :", sys.version.split()[0])
print("PyTorch :", torch.__version__)
print("CUDA visible :", torch.cuda.is_available())
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-pytorch-")
os.chdir(_quiz_temporary_directory.name)
assert not torch.cuda.is_available(), 'Choisir un noyau CPU neuf pour ce notebook.'

Python : 3.11.9
PyTorch : 2.10.0+cpu
CUDA visible : False


## Nous lisons le programme

**1. Une valeur réglable**

requires_grad=True demande de suivre ce poids pendant les opérations différentiables.

**2. Une sortie puis un coût**

La multiplication produit 6 ; le carré mesure son éloignement de zéro.

**3. Demander les dérivées**

backward remonte le calcul effectué. La valeur 24 décrit ici la variation de la perte par rapport au poids.

In [2]:
import torch
w = torch.tensor(3., requires_grad=True)
y = w * 2
perte = (y - 0) ** 2
perte.backward()
print(y.item(), perte.item(), w.grad.item())

6.0 36.0 24.0


## Interpréter avant de modifier

Exécution CPU vérifiée ; les sorties sont conservées dans scripts/verify/pytorch-additions-results.json.

torch porte les tableaux ; les autres paquets organisent leur utilisation.

Le graphe suit les opérations réellement exécutées.

Calculer un gradient et modifier un paramètre sont deux étapes distinctes.

## Exercice 1 — Faire suivre un calcul par autograd

Nous multiplions un poids par trois. Complétons le calcul de sa dérivée, sans changer le poids.

Copions ce squelette dans une nouvelle cellule, puis remplaçons les `...`.

```python
import torch
p = torch.tensor(2., requires_grad=...)
sortie = p * 3
cout = sortie.square()
cout....()
print(p.grad.item())
```

**Piste :** La dérivée de (3p)² est 18p. La méthode backward remplit le gradient.

## Exercice 2 — Distinguer dérivée et mise à jour

Le gradient est calculé. Complétons une mise à jour de descente de gradient, avec un taux de 0,1.

Copions ce squelette dans une nouvelle cellule, puis remplaçons les `...`.

```python
p = torch.tensor(2., requires_grad=True)
(p.square()).backward()
with torch....():
    p -= ... * p.grad
print(p.item())
```

**Piste :** backward ne change pas le poids. La soustraction constitue une opération séparée.

In [3]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()
print('Programme terminé ; fichiers temporaires supprimés.')

Programme terminé ; fichiers temporaires supprimés.


## Solutions — ouvrons-les après notre recherche

Les solutions utilisent leurs propres variables. Nous pouvons les copier dans de nouvelles cellules.

<details>
<summary>Afficher la solution 1 — Faire suivre un calcul par autograd</summary>

<pre><code>import torch
p_exo = torch.tensor(2., requires_grad=True)
sortie_exo = p_exo * 3
cout_exo = sortie_exo.square()
cout_exo.backward()
assert p_exo.grad.item() == 36.0
print(&#x27;gradient :&#x27;, p_exo.grad.item())</code></pre>

</details>

<details>
<summary>Afficher la solution 2 — Distinguer dérivée et mise à jour</summary>

<pre><code>import torch
p_exo = torch.tensor(2., requires_grad=True)
p_exo.square().backward()
avant_exo = p_exo.detach().clone()
with torch.no_grad():
    p_exo -= 0.1 * p_exo.grad
assert torch.allclose(p_exo, torch.tensor(1.6))
assert avant_exo.item() == 2.0
print(&#x27;poids avant et après :&#x27;, avant_exo.item(), p_exo.item())</code></pre>

</details>

